# Dijet pointing-resolution systematic
Estimate the one-sided pointing uncertainty `abs(Reco/Ref - 1)` in the CM and Lab frames. Full shapes are unit-normalized. The CM workflow folds whole positive-eta bins and their negative-eta reflections from each unnormalized projection, with independent errors; no Lab F/B is constructed.

Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

Display comparisons default to ROOT independent-error propagation. Shared-event and normalization correlations are not included. Binomial errors require an actual numerator-subset efficiency and are not appropriate for unfolded, smeared, normalized-shape, or F/B-to-F/B comparisons.

<!-- detailed-workflow-guide -->

### Detailed workflow and systematic construction

This is a one-variation estimate, displayed as a symmetric uncertainty: $s_i=|R_i-1|$, where $R_i=Reco_i/Ref_i$. CM and Lab full shapes are independently normalized; only CM has an F/B estimator.

`SYSTEMATIC_EXTRACTION` chooses fitted or raw-bin ratio values. Fits are evaluated at bin centers within the accepted range, with overlapping boundary bins clamped to its endpoints. Optional outward running-maximum smoothing is applied afterward. The retained uncertainty is fractional; percent figures multiply display clones by 100, while ratio bands show $1 \pm s_i$.

F/B is constructed from unnormalized hemisphere yields using independent errors. Shared-event and normalization correlations are not included in the comparison errors. Fit functions, ranges, weights, coefficients, and covariance remain controlled by the existing analysis configuration.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2
from dataclasses import replace
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / 'CMakeLists.txt').is_file()
        and (candidate / 'hist_analysis').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL,
    STANDARD_DIJET_ETA_CUT_INDEX,
    TEST_DIJET_PTAVE_BINS,
    DIJET_PTAVE_BINS,
)
from hist_analysis.python.data_distributions import (
    forward_backward_from_full, project_data_histogram,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import (
    normalize_histogram, ratio_to_nominal,
)
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.root_style import (
    COLORS, DEFAULT_PLOT_STYLE, save_canvas,
)
from hist_analysis.python.systematic_fits import (
    calculate_one_sided_systematic,
    fit_histogram_variations,
    format_fit_summary_lines,
    smooth_systematic_running_max,
    write_systematic_csv,
)

ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = True  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
# Use one CMS renderer for fits, bands, annotations and original markers.
configure_mplhep_export(False)
from functools import partial
from hist_analysis.python.systematic_plotting import (
    save_systematic_canvas,
    build_systematic_band as make_systematic_band,
    draw_systematic_overlay,
)
_export_options = dict(mplhep=SAVE_MPLHEP, mplhep_png=SAVE_MPLHEP_PNG,
                       cms_header_left='CMS Simulation')
save_canvas = partial(save_systematic_canvas, **_export_options)
draw_overlay = partial(draw_systematic_overlay, **_export_options)

from hist_analysis.python.systematic_methods import (
    construct_systematic_variations, validate_systematic_method,
)
from hist_analysis.python.systematic_plotting import draw_systematic_variation_overlay

from hist_analysis.python.systematic_diagnostics import make_systematic_diagnostics


## Configuration
Comparison ratios allow standard (`''`) or binomial (`'B'`) errors. `FORWARD_BACKWARD_RATIO_OPTION` is protected: F/B construction is never binomial.

`REFERENCE_DEFINITION` selects the reference pT coordinate for both CM and Lab. The default `refWithRecoPtAve` retains `(recoPtAve, refEta)`; `refWithRefPtAve` selects `(refPtAve, refEta)` for the same matched reco-selected pair. Reco remains binned in reco pTave. Both reference definitions apply the same reference-jet eta acceptance; Reco applies reco-jet acceptance. Reference jets are neither re-ranked nor subjected to additional reference-pT or reference-delta-phi cuts. Thus the alternate ratio includes pT-bin migration as well as pointing effects. New inputs must be regenerated; missing keys raise an error. Output filenames contain `RecoPtAve` for `(recoPtAve, refDijetEta)` or `RefPtAve` for `(refPtAve, refDijetEta)`.

Outputs are stored under the configured output root / `FILE_STEM` (`jetId`, `trkMax`, or `noSel`), and filenames include the same selection. Existing binning and MPLHEP subdirectories are retained.

To run: choose `REFERENCE_DEFINITION`, `GENERATOR`, `DIRECTION`, and `FILE_STEM` in the configuration cell, then restart the kernel and run all cells from the top. The configuration prints the resolved input file, output directory, and exact CM/Lab keys. The literal reference setting takes precedence over any shell environment variable.

`PREFER_PRODUCED_INPUT = False` reads the original file directly. Use this when the original has been regenerated but the reusable files are stale. Set it to `True` after regenerating reusable inputs.

`SYSTEMATIC_METHOD` selects `direct_ratio` (legacy default) or `folded_deviation`, supplying `1 + abs(1 - R)` to the existing fit or raw-bin estimator. Symmetric ratio errors are retained; zero-content/zero-error empty sentinels stay empty and nonfinite folded inputs fail. `SYSTEMATIC_EXTRACTION`, combination (where supported), and running-maximum smoothing remain independent. Folded outputs append `_method_folded_deviation`; raw exports append `_extraction_bin_by_bin`. Direct-fit filenames and the headerless four-column CSV schema remain unchanged. `SYSTEMATIC_Y_RANGE=(0.0,5.0)` fixes all percent uncertainty plots to 0–5%; change this tuple or use `None` for automatic limits.


`STAT_UNCERTAINTY_SCALE` (default `1.0`) scales only statistical error bars on the full and F/B variation/default plots, for both direct ratios and `1 + |1 - variation/default|`, including their systematic-band overlays. Plotting uses fresh copies after extraction and smoothing. Original eta and F/B distribution overlays, analysis ratio errors, fits, systematic bands, diagnostics, and CSV outputs remain unchanged. Use a finite positive number; rerunning always starts from unscaled analysis histograms.


In [ ]:
from hist_analysis.config.histograms import systematic_pt_bins
# Presentation only: applied to direct/folded variation ratios after extraction.
from hist_analysis.python.systematic_plotting import validate_stat_uncertainty_scale
STAT_UNCERTAINTY_SCALE = 1.0
validate_stat_uncertainty_scale(STAT_UNCERTAINTY_SCALE)

DIJET_BINNING = 'normal'  # normal or test
ETA_CUT_VALUE = 1.9  # per-jet |etaCM| <= value; supported producer cuts only
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Optional plots only; these switches do not select nominal results or CSVs.
PLOT_SMOOTHED_VS_UNSMOOTHED = False
PLOT_BINOMIAL_VS_STANDARD_ERRORS = True
SYSTEMATIC_METHOD = 'folded_deviation'  # direct_ratio or folded_deviation
validate_systematic_method(SYSTEMATIC_METHOD)
REFERENCE_DEFINITION = 'refWithRecoPtAve'  # change only this line to select the reference
# refWithRecoPtAve (legacy) or refWithRefPtAve; Reco always uses reco pTave.
# Input histogram suffixes preserve the producer's existing ROOT keys.
REFERENCE_HISTOGRAM_SUFFIX = {
    'refWithRecoPtAve': '',
    'refWithRefPtAve': 'RefPtAve',
}
if REFERENCE_DEFINITION not in REFERENCE_HISTOGRAM_SUFFIX:
    raise ValueError(f'Unknown REFERENCE_DEFINITION: {REFERENCE_DEFINITION!r}; choose {tuple(REFERENCE_HISTOGRAM_SUFFIX)}')
REF_PT_LABEL = 'reco' if REFERENCE_DEFINITION == 'refWithRecoPtAve' else 'ref'
REFERENCE_OUTPUT_TAG = {
    'refWithRecoPtAve': 'RecoPtAve',
    'refWithRefPtAve': 'RefPtAve',
}[REFERENCE_DEFINITION]

GENERATOR = 'embedding'  # embedding or pythia
DIRECTION = 'combined'   # combined, Pbgoing, or pgoing
FILE_STEM = 'jetId'  # jetId, trkMax, or noSel
PREFER_PRODUCED_INPUT = False  # False: original ROOT file; True: reusable file when available
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUTS = DIJET_ETA_CUTS  # shared current production cut order
# ETA_CUT_INDEX = eta_cut_index(ETA_CUT_VALUE)
ETA_CUT_INDEX = eta_cut_index(ETA_CUT_VALUE)
LAB_ETA_CUT_INDEX = DIJET_ETA_CUTS.index(2.4)
# PTAVE_BINS = DIJET_PTAVE_BINS
PTAVE_BINS = tuple(systematic_pt_bins(DIJET_BINNING))
REBIN_ETA = 2

SYSTEMATIC_EXTRACTION = 'fit'  # fit or bin_by_bin
APPLY_SYSTEMATIC_SMOOTHING = True
FULL_SMOOTHING_ORIGIN = -0.465 + 0.00001

FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: never binomial
FULL_COMPARISON_RATIO_OPTION = 'B'  # ROOT binomial prescription; a subset interpretation is not established
FB_COMPARISON_RATIO_OPTION = 'B'  # ROOT binomial prescription for F/B comparisons, not F/B construction

FULL_FIT_FUNCTION = 'pol2'
FB_FIT_FUNCTION = 'pol1'
FULL_FIT_INITIAL_VALUES = {
    'Reco / Ref': (1.0, 0.0, 0.0),
    # 'Reco / Ref': (1.0, 0.0, 0.0, 0.0, 0.0),
}
FB_FIT_INITIAL_VALUES = {
    '(F/B) Reco / Ref': (1.0, 0.0),
}
FIT_OPTIONS = 'RQS0'          # range, result, quiet, no draw
FIT_WEIGHT_OPTION = ''       # 'W': unit weights; '': use bin errors
EFFECTIVE_FIT_OPTIONS = FIT_OPTIONS + FIT_WEIGHT_OPTION
FIT_WEIGHT_TAG = 'weights1' if FIT_WEIGHT_OPTION == 'W' else 'weightsStd'
OUTPUT_CONFIGURATION_TAG = (
    f'{FILE_STEM}_'
    f'fullFit_{FULL_FIT_FUNCTION}_fbFit_{FB_FIT_FUNCTION}'
    f'_{FIT_WEIGHT_TAG}_systCombMax_{REFERENCE_OUTPUT_TAG}'
)
if SYSTEMATIC_METHOD != 'direct_ratio':
    OUTPUT_CONFIGURATION_TAG += f'_method_{SYSTEMATIC_METHOD}'
if SYSTEMATIC_EXTRACTION != 'fit':
    OUTPUT_CONFIGURATION_TAG += f'_extraction_{SYSTEMATIC_EXTRACTION}'
SHOW_FIT_RESULTS = True

DRAW_GRID = True
SAVE_PNG = False
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
SYSTEMATIC_Y_RANGE = (0.0, 5.0)  # percent; all ptAve bins; None for automatic limits
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_POINTING_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_pointing_resolution',
)) / FILE_STEM

if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError('GENERATOR must be embedding or pythia')
if DIRECTION not in ('combined', 'Pbgoing', 'pgoing'):
    raise ValueError('DIRECTION must be combined, Pbgoing, or pgoing')
if SYSTEMATIC_EXTRACTION not in ('fit', 'bin_by_bin'):
    raise ValueError("SYSTEMATIC_EXTRACTION must be 'fit' or 'bin_by_bin'")
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('F/B construction must use independent errors')
if FIT_WEIGHT_OPTION not in ('', 'W'):
    raise ValueError("FIT_WEIGHT_OPTION must be empty or 'W'")
for option_name, option in (
    ('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
    ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f'{option_name} must be empty or B')
for name, index in (('ETA_CUT_INDEX', ETA_CUT_INDEX), ('LAB_ETA_CUT_INDEX', LAB_ETA_CUT_INDEX)):
    if not 0 <= index < len(ETA_CUTS):
        raise IndexError(f'Invalid {name}: {index}')

ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
LAB_ETA_CUT = ETA_CUTS[LAB_ETA_CUT_INDEX]
if DIRECTION == 'combined':
    INPUT_FILE = resolve_combined_file(
        BASE_DIR, GENERATOR, FILE_STEM, prefer_produced=PREFER_PRODUCED_INPUT,
    )
else:
    INPUT_FILE = resolve_direction_file(
        BASE_DIR, GENERATOR, DIRECTION, FILE_STEM,
        prefer_produced=PREFER_PRODUCED_INPUT,
    )
if not INPUT_FILE.exists():
    raise FileNotFoundError(INPUT_FILE)

HISTOGRAM_KEYS = {
    'Reco': f'hRecoDijetPtEtaCMPointingRes_{ETA_CUT_INDEX}',
    'Ref': f'hRefDijetPtEtaCMPointingRes{REFERENCE_HISTOGRAM_SUFFIX[REFERENCE_DEFINITION]}_{ETA_CUT_INDEX}',
}
LAB_HISTOGRAM_KEYS = {
    'Reco': f'hRecoDijetPtEtaLabPointingRes_{LAB_ETA_CUT_INDEX}',
    'Ref': f'hRefDijetPtEtaLabPointingRes{REFERENCE_HISTOGRAM_SUFFIX[REFERENCE_DEFINITION]}_{LAB_ETA_CUT_INDEX}',
}
STYLE = replace(
    DEFAULT_PLOT_STYLE,
    annotation_text_size=0.026,
    annotation_line_spacing=0.039,
    legend_text_size=0.028,
)
print(f'Reference: {REFERENCE_DEFINITION} (filename tag: {REFERENCE_OUTPUT_TAG})')
print(f'Input: {INPUT_FILE}')
print(f'Output: {OUTPUT_DIR}')
print('CM keys:', HISTOGRAM_KEYS)
print('Lab keys:', LAB_HISTOGRAM_KEYS)
INPUT_FILE
draw_variation_overlay = partial(
    draw_systematic_variation_overlay, method=SYSTEMATIC_METHOD, renderer=draw_overlay,
    stat_uncertainty_scale=STAT_UNCERTAINTY_SCALE,
)

# Output identity uses the actual configured selections. Each execution gets
# dated renderer directories and unique filenames; earlier files are preserved.
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection, nominal_jet_eta_cut
OUTPUT_SELECTIONS = OutputSelection(FILE_STEM, {'CM': ETA_CUT, 'Lab': LAB_ETA_CUT}, PTAVE_BINS, 'ptAve')
OUTPUT_DIR = OutputDirectory(OUTPUT_DIR, OUTPUT_SELECTIONS, workflow_root=Path(os.environ.get('DIJET_POINTING_SYSTEMATICS_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_pointing_resolution')))



In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
def draw_ratio_with_systematic_band(
    ratios, systematic, fit_functions, *, x_range, y_range,
    x_title, y_title, annotations, output, canvas_name, evaluation_range,
):
    canvas = draw_variation_overlay(
        ratios, title='', x_title=x_title, y_title=y_title,
        x_range=x_range, y_range=y_range, reference_y=1.0,
        annotations=annotations, grid=DRAW_GRID,
        overlay_functions=fit_functions,
        style_indices={next(iter(ratios)): 0}, style=STYLE,
        output=None, canvas_name=canvas_name,
    )
    graph = make_systematic_band(systematic,
        evaluation_range=evaluation_range,
        name=f'{canvas_name}_systematic_band')
    graph.Draw('E2 SAME')
    for histogram in canvas._systematic_plot_histograms.values():
        histogram.Draw('E1 SAME')
    for function in fit_functions.values():
        function.Draw('SAME')

    legend = canvas._overlay_objects[0]
    legend.AddEntry(graph, 'Systematic uncertainty', 'f')
    canvas._overlay_objects.append(graph)
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    return canvas

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
pointing_results = {}
eta_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_range = (0.0, ETA_CUT + 0.1)

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    tag = (
        f'{GENERATOR}_{DIRECTION}_pointing_etaCM_{round(10 * ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )
    output_prefix = f'{GENERATOR}_{DIRECTION}_pointing'
    output_suffix = (
        f'{OUTPUT_CONFIGURATION_TAG}_etaCM_{round(10 * ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )

    raw_eta = {
        label: project_data_histogram(
            INPUT_FILE, key, 'eta', ptave_range,
            name=f'h_{tag}_{label.lower()}_raw',
            rebin=REBIN_ETA, normalization='none',
        )
        for label, key in HISTOGRAM_KEYS.items()
    }
    eta_shapes = {
        label: normalize_histogram(histogram, 'integral')
        for label, histogram in raw_eta.items()
    }
    fb_ratios = {
        label: forward_backward_from_full(
            histogram, name=f'h_{tag}_{label.lower()}_fb',
        )
        for label, histogram in raw_eta.items()
    }

    eta_ratios = {
        'Reco / Ref': ratio_to_nominal(
            eta_shapes['Reco'], eta_shapes['Ref'],
            name=f'h_{tag}_reco_ref', option=FULL_COMPARISON_RATIO_OPTION,
        )
    }
    fb_comparisons = {
        '(F/B) Reco / Ref': ratio_to_nominal(
            fb_ratios['Reco'], fb_ratios['Ref'],
            name=f'h_{tag}_fb_reco_ref', option=FB_COMPARISON_RATIO_OPTION,
        )
    }

    eta_raw_ratios = eta_ratios
    eta_ratios = construct_systematic_variations(eta_raw_ratios, method=SYSTEMATIC_METHOD)
    fb_raw_comparisons = fb_comparisons
    fb_comparisons = construct_systematic_variations(fb_raw_comparisons, method=SYSTEMATIC_METHOD)
    if SYSTEMATIC_EXTRACTION == 'fit':
        eta_fits, eta_fit_summary = fit_histogram_variations(
            eta_ratios, formula=FULL_FIT_FUNCTION,
            fit_range=(-ETA_CUT, ETA_CUT), name_prefix=f'f_{tag}_full',
            fit_options=EFFECTIVE_FIT_OPTIONS,
            initial_values=FULL_FIT_INITIAL_VALUES,
        )
        fb_fits, fb_fit_summary = fit_histogram_variations(
            fb_comparisons, formula=FB_FIT_FUNCTION,
            fit_range=(0.0, ETA_CUT), name_prefix=f'f_{tag}_fb',
            fit_options=EFFECTIVE_FIT_OPTIONS,
            initial_values=FB_FIT_INITIAL_VALUES,
        )
    else:
        eta_fits, eta_fit_summary = {}, {}
        fb_fits, fb_fit_summary = {}, {}
    use_fit = SYSTEMATIC_EXTRACTION == 'fit'
    eta_systematic_unsmoothed = calculate_one_sided_systematic(
        eta_ratios['Reco / Ref'], name=f'h_{tag}_full_syst',
        variation_function=eta_fits['Reco / Ref'] if use_fit else None,
        evaluation_range=(-ETA_CUT, ETA_CUT),
    )
    fb_systematic_unsmoothed = calculate_one_sided_systematic(
        fb_comparisons['(F/B) Reco / Ref'], name=f'h_{tag}_fb_syst',
        variation_function=(
            fb_fits['(F/B) Reco / Ref'] if use_fit else None
        ),
        evaluation_range=(0.0, ETA_CUT),
    )

    eta_systematic = (
        smooth_systematic_running_max(
            eta_systematic_unsmoothed,
            name=f'{eta_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(-ETA_CUT, ETA_CUT),
            smoothing_origin=FULL_SMOOTHING_ORIGIN,
        )
        if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_unsmoothed
    )
    fb_systematic = (
        smooth_systematic_running_max(
            fb_systematic_unsmoothed,
            name=f'{fb_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(0.0, ETA_CUT),
        )
        if APPLY_SYSTEMATIC_SMOOTHING else fb_systematic_unsmoothed
    )

    eta_percent = eta_systematic.Clone(f'{eta_systematic.GetName()}_percent')
    eta_percent.SetDirectory(0)
    eta_percent.Scale(100.0)
    fb_percent = fb_systematic.Clone(f'{fb_systematic.GetName()}_percent')
    fb_percent.SetDirectory(0)
    fb_percent.Scale(100.0)

    annotations = (
        'Embedding MC' if GENERATOR == 'embedding' else 'PYTHIA 8',
        {'combined': 'Combined', 'pgoing': 'p-going', 'Pbgoing': 'Pb-going'}[DIRECTION],
        'Reference-matched dijets',
        f'{low:g} #leq p_{{T}}^{{ave,reco}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| #leq {ETA_CUT:g}',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    common_plot_options = dict(
        title='', annotations=annotations, grid=DRAW_GRID,
        style=STYLE, save_png=SAVE_PNG,
    )
    canvases = {}
    canvases['eta_overlay'] = draw_overlay(
        eta_shapes, x_title='#eta_{CM}^{dijet}',
        y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_range,
        style_indices={'Reco': 0, 'Ref': 1},
        output=OUTPUT_DIR / f'{output_prefix}_full_overlay_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_overlay', **common_plot_options,
    )
    canvases['eta_ratio'] = draw_variation_overlay(
        eta_ratios, x_title='#eta_{CM}^{dijet}', y_title='Reco / Ref',
        x_range=eta_range, y_range=FULL_RATIO_RANGE, reference_y=1.0,
        overlay_functions=eta_fits,
        overlay_text=(
            format_fit_summary_lines(eta_fit_summary)
            if SHOW_FIT_RESULTS else None
        ),
        style_indices={'Reco / Ref': 0},
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_ratio', **common_plot_options,
    )
    canvases['fb_overlay'] = draw_overlay(
        fb_ratios, x_title='|#eta_{CM}^{dijet}|',
        y_title='Forward / Backward', x_range=fb_range, y_range=FB_RANGE,
        reference_y=1.0, style_indices={'Reco': 0, 'Ref': 1},
        output=OUTPUT_DIR / f'{output_prefix}_fb_overlay_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_overlay', **common_plot_options,
    )
    canvases['fb_ratio'] = draw_variation_overlay(
        fb_comparisons, x_title='|#eta_{CM}^{dijet}|',
        y_title='(F/B)_{Reco} / (F/B)_{Ref}', x_range=fb_range,
        y_range=FB_DOUBLE_RATIO_RANGE, reference_y=1.0,
        overlay_functions=fb_fits,
        overlay_text=(
            format_fit_summary_lines(fb_fit_summary)
            if SHOW_FIT_RESULTS else None
        ),
        style_indices={'(F/B) Reco / Ref': 0},
        output=OUTPUT_DIR / f'{output_prefix}_fb_ratio_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_ratio', **common_plot_options,
    )

    suffix = 'smoothed' if APPLY_SYSTEMATIC_SMOOTHING else 'nonsmoothed'
    eta_systematic_csv = write_systematic_csv(
        eta_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_relative_{output_suffix}.csv',
        evaluation_range=(-ETA_CUT, ETA_CUT),
    )
    fb_systematic_csv = write_systematic_csv(
        fb_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_fb_relative_{output_suffix}.csv',
        evaluation_range=(0.0, ETA_CUT),
    )
    canvases['eta_syst'] = draw_overlay(
        {'Pointing resolution': eta_percent},
        x_title='#eta_{CM}^{dijet}', y_title='Pointing Rel. Syst. Uncrt. (%)',
        x_range=eta_range, y_range=SYSTEMATIC_Y_RANGE, show_legend=True,
        output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_syst', **common_plot_options,
    )
    canvases['fb_syst'] = draw_overlay(
        {'Pointing resolution': fb_percent},
        x_title='|#eta_{CM}^{dijet}|', y_title='Pointing Rel. Syst. Uncrt. (%)',
        x_range=fb_range, y_range=SYSTEMATIC_Y_RANGE, show_legend=True,
        output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_fb_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_syst', **common_plot_options,
    )
    canvases['eta_band'] = draw_ratio_with_systematic_band(
        eta_ratios, eta_systematic, eta_fits, x_range=eta_range,
        evaluation_range=(-ETA_CUT, ETA_CUT),
        y_range=FULL_RATIO_RANGE, x_title='#eta_{CM}^{dijet}',
        y_title='Reco / Ref', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_band',
    )
    canvases['fb_band'] = draw_ratio_with_systematic_band(
        fb_comparisons, fb_systematic, fb_fits, x_range=fb_range,
        evaluation_range=(0.0, ETA_CUT),
        y_range=FB_DOUBLE_RATIO_RANGE, x_title='|#eta_{CM}^{dijet}|',
        y_title='(F/B)_{Reco} / (F/B)_{Ref}', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_fb_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_band',
    )

    pointing_results[ptave_range] = {
        'systematic_method': SYSTEMATIC_METHOD, 'systematic_extraction': SYSTEMATIC_EXTRACTION,
        'raw_eta': raw_eta, 'eta_shapes': eta_shapes,
        'selected_keys': dict(HISTOGRAM_KEYS),
        'forward_backward': fb_ratios, 'eta_raw_ratios': eta_raw_ratios, 'eta_ratios': eta_ratios,
        'fb_raw_comparisons': fb_raw_comparisons,
        'fb_ratios': fb_comparisons, 'eta_fits': eta_fits,
        'fb_fits': fb_fits, 'eta_fit_summary': eta_fit_summary,
        'fb_fit_summary': fb_fit_summary,
        'eta_systematic_unsmoothed': eta_systematic_unsmoothed,
        'fb_systematic_unsmoothed': fb_systematic_unsmoothed,
        'eta_systematic': eta_systematic,
        'fb_systematic': fb_systematic,
        'eta_systematic_csv': eta_systematic_csv,
        'fb_systematic_csv': fb_systematic_csv,
        'annotations': annotations, 'canvases': canvases,
    }
    print(ptave_range, eta_fit_summary, fb_fit_summary)
    for canvas in canvases.values():
        display(canvas)

## Lab-frame pointing-resolution estimator
The Lab estimator uses its own eta-cut index and full distributions only. No Lab Forward/Backward ratios are constructed.

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
lab_pointing_results = {}
lab_eta_range = (-LAB_ETA_CUT - 0.1, LAB_ETA_CUT + 0.1)

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    tag = (
        f'{GENERATOR}_{DIRECTION}_pointing_etaLab_'
        f'{round(10 * LAB_ETA_CUT):g}_ptave_{ptave_tag}'
    )
    output_prefix = f'{GENERATOR}_{DIRECTION}_pointing'
    output_suffix = (
        f'{OUTPUT_CONFIGURATION_TAG}_etaLab_{round(10 * LAB_ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )

    raw_eta = {
        label: project_data_histogram(
            INPUT_FILE, key, 'eta', ptave_range,
            name=f'h_{tag}_{label.lower()}_raw',
            rebin=REBIN_ETA, normalization='none',
        )
        for label, key in LAB_HISTOGRAM_KEYS.items()
    }
    eta_shapes = {
        label: normalize_histogram(histogram, 'integral')
        for label, histogram in raw_eta.items()
    }
    eta_ratios = {
        'Reco / Ref': ratio_to_nominal(
            eta_shapes['Reco'], eta_shapes['Ref'],
            name=f'h_{tag}_reco_ref', option=FULL_COMPARISON_RATIO_OPTION,
        )
    }
    eta_raw_ratios = eta_ratios
    eta_ratios = construct_systematic_variations(eta_raw_ratios, method=SYSTEMATIC_METHOD)
    if SYSTEMATIC_EXTRACTION == 'fit':
        eta_fits, eta_fit_summary = fit_histogram_variations(
            eta_ratios, formula=FULL_FIT_FUNCTION,
            fit_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
            name_prefix=f'f_{tag}_full', fit_options=EFFECTIVE_FIT_OPTIONS,
            initial_values=FULL_FIT_INITIAL_VALUES,
        )
    else:
        eta_fits, eta_fit_summary = {}, {}
    use_fit = SYSTEMATIC_EXTRACTION == 'fit'
    eta_systematic_unsmoothed = calculate_one_sided_systematic(
        eta_ratios['Reco / Ref'], name=f'h_{tag}_full_syst',
        variation_function=eta_fits['Reco / Ref'] if use_fit else None,
        evaluation_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
    )
    eta_systematic = (
        smooth_systematic_running_max(
            eta_systematic_unsmoothed,
            name=f'{eta_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
            smoothing_origin=0.0,
        )
        if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_unsmoothed
    )
    eta_percent = eta_systematic.Clone(f'{eta_systematic.GetName()}_percent')
    eta_percent.SetDirectory(0)
    eta_percent.Scale(100.0)
    annotations = (
        'Embedding MC' if GENERATOR == 'embedding' else 'PYTHIA 8',
        {'combined': 'Combined', 'pgoing': 'p-going', 'Pbgoing': 'Pb-going'}[DIRECTION],
        'Reference-matched dijets',
        f'{low:g} #leq p_{{T}}^{{ave,reco}} < {high:g} GeV',
        f'|#eta_{{Lab}}^{{jet}}| #leq {LAB_ETA_CUT:g}',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    common_plot_options = dict(
        title='', annotations=annotations, grid=DRAW_GRID,
        style=STYLE, save_png=SAVE_PNG,
    )
    suffix = 'smoothed' if APPLY_SYSTEMATIC_SMOOTHING else 'nonsmoothed'
    eta_systematic_csv = write_systematic_csv(
        eta_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_relative_{output_suffix}.csv',
        evaluation_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
    )
    canvases = {
        'eta_overlay': draw_overlay(
            eta_shapes, x_title='#eta_{Lab}^{dijet}',
            y_title='1/N dN/d#eta_{Lab}^{dijet}', x_range=lab_eta_range,
            style_indices={'Reco': 0, 'Ref': 1},
            output=OUTPUT_DIR / f'{output_prefix}_full_overlay_{output_suffix}.pdf',
            canvas_name=f'{tag}_full_overlay', **common_plot_options,
        ),
        'eta_ratio': draw_variation_overlay(
            eta_ratios, x_title='#eta_{Lab}^{dijet}', y_title='Reco / Ref',
            x_range=lab_eta_range, y_range=FULL_RATIO_RANGE, reference_y=1.0,
            overlay_functions=eta_fits,
            overlay_text=(format_fit_summary_lines(eta_fit_summary) if SHOW_FIT_RESULTS else None),
            style_indices={'Reco / Ref': 0},
            output=OUTPUT_DIR / f'{output_prefix}_full_ratio_{output_suffix}.pdf',
            canvas_name=f'{tag}_full_ratio', **common_plot_options,
        ),
        'eta_syst': draw_overlay(
            {'Pointing resolution': eta_percent},
            x_title='#eta_{Lab}^{dijet}',
            y_title='Pointing Rel. Syst. Uncrt. (%)',
            x_range=lab_eta_range, y_range=SYSTEMATIC_Y_RANGE,
            show_legend=True,
            output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_{output_suffix}.pdf',
            canvas_name=f'{tag}_full_syst', **common_plot_options,
        ),
    }
    canvases['eta_band'] = draw_ratio_with_systematic_band(
        eta_ratios, eta_systematic, eta_fits, x_range=lab_eta_range,
        evaluation_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
        y_range=FULL_RATIO_RANGE, x_title='#eta_{Lab}^{dijet}',
        y_title='Reco / Ref', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_band',
    )
    lab_pointing_results[ptave_range] = {
        'systematic_method': SYSTEMATIC_METHOD, 'systematic_extraction': SYSTEMATIC_EXTRACTION,
        'raw_eta': raw_eta, 'eta_shapes': eta_shapes,
        'selected_keys': dict(LAB_HISTOGRAM_KEYS),
        'eta_raw_ratios': eta_raw_ratios, 'eta_ratios': eta_ratios, 'eta_fits': eta_fits,
        'eta_fit_summary': eta_fit_summary,
        'eta_systematic_unsmoothed': eta_systematic_unsmoothed,
        'eta_systematic': eta_systematic,
        'eta_systematic_csv': eta_systematic_csv, 'annotations': annotations, 'canvases': canvases,
    }
    print('Lab', ptave_range, eta_fit_summary)
    for canvas in canvases.values():
        display(canvas)

In [ ]:
# Cell role: perform analysis step 6.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
for ptave_range, result in pointing_results.items():
    normalized_integrals = {
        label: histogram.Integral()
        for label, histogram in result['eta_shapes'].items()
    }
    assert all(
        abs(integral - 1.0) < 1e-9
        for integral in normalized_integrals.values()
    ), normalized_integrals
    for histogram in result['forward_backward'].values():
        assert not histogram.GetDirectory()
    print(
        ptave_range, 'normalized integrals', normalized_integrals,
        'selected keys', result['selected_keys'],
    )

for ptave_range, result in lab_pointing_results.items():
    normalized_integrals = {
        label: histogram.Integral()
        for label, histogram in result['eta_shapes'].items()
    }
    assert all(
        abs(integral - 1.0) < 1e-9
        for integral in normalized_integrals.values()
    ), normalized_integrals
    print(
        'Lab', ptave_range, 'normalized integrals', normalized_integrals,
        'selected keys', result['selected_keys'],
    )

## Optional systematic diagnostic overlays

Enable `PLOT_SMOOTHED_VS_UNSMOOTHED` and/or `PLOT_BINOMIAL_VS_STANDARD_ERRORS` in the configuration cell. Both default to `False`. These figures compare final fractional estimates on percent display clones; nominal histograms and CSV exports remain unchanged. Smoothing is the existing outward running maximum **after** the estimator. The error-treatment comparison changes only variation/default TH1 division (`B` versus standard); F/B construction stays standard. The selected method, fit/raw mode, fits, estimator and nominal smoothing are retained. ROOT `B` is an efficiency-error prescription; these systematic ratios are not demonstrated binomial efficiencies. Diagnostic figures have `_diagnostic_` filenames and write no CSVs.


In [ ]:
# Diagnostic-only stage: all nominal calculations and CSV exports are complete.
if PLOT_SMOOTHED_VS_UNSMOOTHED or PLOT_BINOMIAL_VS_STANDARD_ERRORS:
    for diagnostic_frame, diagnostic_results in (('CM', pointing_results), ('Lab', lab_pointing_results)):
        for diagnostic_interval, diagnostic_result in diagnostic_results.items():
            cut = ETA_CUT if diagnostic_frame == 'CM' else LAB_ETA_CUT
            diagnostic_inputs = [
                ('full', diagnostic_result['eta_shapes'], (-cut, cut),
                 FULL_SMOOTHING_ORIGIN if diagnostic_frame == 'CM' else 0.0,
                 f'#eta_{{{diagnostic_frame}}}^{{dijet}}', (-cut - 0.1, cut + 0.1),
                 FULL_FIT_FUNCTION, FULL_FIT_INITIAL_VALUES, 'Reco / Ref'),
            ]
            if diagnostic_frame == 'CM':
                diagnostic_inputs.append(
                    ('fb', diagnostic_result['forward_backward'], (0.0, cut), None,
                     '|#eta_{CM}^{dijet}|', (0.0, cut + 0.1),
                     FB_FIT_FUNCTION, FB_FIT_INITIAL_VALUES, '(F/B) Reco / Ref'))
            diagnostic_result['diagnostics'] = {}
            for observable, shapes, acceptance, origin, title, x_limits, formula, seeds, label in diagnostic_inputs:
                key = 'eta' if observable == 'full' else 'fb'
                diagnostic_result['diagnostics'][observable] = make_systematic_diagnostics(
                    unsmoothed=diagnostic_result[f'{key}_systematic_unsmoothed'],
                    nominal=diagnostic_result[f'{key}_systematic'],
                    numerators={label: shapes['Reco']}, denominator=shapes['Ref'],
                    nominal_csv=diagnostic_result[f'{key}_systematic_csv'],
                    annotations=diagnostic_result['annotations'], source='Pointing resolution',
                    observable=f'{diagnostic_frame} ' + ('full' if observable == 'full' else 'F/B'),
                    plot_smoothed_vs_unsmoothed=PLOT_SMOOTHED_VS_UNSMOOTHED,
                    plot_binomial_vs_standard_errors=PLOT_BINOMIAL_VS_STANDARD_ERRORS,
                    method=SYSTEMATIC_METHOD, extraction=SYSTEMATIC_EXTRACTION,
                    formula=formula, fit_options=EFFECTIVE_FIT_OPTIONS, initial_values=seeds,
                    evaluation_range=acceptance, apply_smoothing=APPLY_SYSTEMATIC_SMOOTHING,
                    smoothing_origin=origin, renderer=draw_overlay, style=STYLE,
                    x_title=title, x_range=x_limits, y_range=SYSTEMATIC_Y_RANGE,
                    grid=DRAW_GRID, save_png=SAVE_PNG,
                )
                for diagnostic in diagnostic_result['diagnostics'][observable].values():
                    display(diagnostic['canvas'])
